# Data Exploration: Food Ingredients and Recipe Dataset\n\nThis notebook explores the raw dataset to understand its structure, quality, and key characteristics before preprocessing.

In [1]:
import pandas as pd
import ast
import os
from pathlib import Path
from collections import Counter

DATA_DIR = Path("../data/raw")
CSV_PATH = DATA_DIR / "Food Ingredients and Recipe Dataset with Image Name Mapping.csv"
IMAGE_DIR = DATA_DIR / "Food Images" / "Food Images"

df = pd.read_csv(CSV_PATH, index_col=0)
print(f"Loaded {len(df):,} rows")

Loaded 13,501 rows


## Shape, Columns, and Data Types

In [2]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:")
print(df.dtypes)

Shape: (13501, 5)

Columns: ['Title', 'Ingredients', 'Instructions', 'Image_Name', 'Cleaned_Ingredients']

Data types:
Title                  object
Ingredients            object
Instructions           object
Image_Name             object
Cleaned_Ingredients    object
dtype: object


## Missing Values

In [3]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal rows with any missing value: {df.isnull().any(axis=1).sum()}")

Missing values per column:
Title                  5
Ingredients            0
Instructions           8
Image_Name             0
Cleaned_Ingredients    0
dtype: int64

Total rows with any missing value: 8


## Sample: Cleaned_Ingredients (10 rows)

In [4]:
for i, row in df.head(10).iterrows():
    print(f"[{i}] {row['Title']}")
    print(f"    {row['Cleaned_Ingredients']}\n")

[0] Miso-Butter Roast Chicken With Acorn Squash Panzanella
    ['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher salt, divided, plus more', '2 small acorn squash (about 3 lb. total)', '2 Tbsp. finely chopped sage', '1 Tbsp. finely chopped rosemary', '6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature', '¼ tsp. ground allspice', 'Pinch of crushed red pepper flakes', 'Freshly ground black pepper', '⅓ loaf good-quality sturdy white bread, torn into 1" pieces (about 2½ cups)', '2 medium apples (such as Gala or Pink Lady; about 14 oz. total), cored, cut into 1" pieces', '2 Tbsp. extra-virgin olive oil', '½ small red onion, thinly sliced', '3 Tbsp. apple cider vinegar', '1 Tbsp. white miso', '¼ cup all-purpose flour', '2 Tbsp. unsalted butter, room temperature', '¼ cup dry white wine', '2 cups unsalted chicken broth', '2 tsp. white miso', 'Kosher salt', 'freshly ground pepper']

[1] Crispy Salt and Pepper Potatoes
    ['2 large egg whites', '1 pound new potatoes (about 1 inch in dia

## Sample: First Row of Each Column

In [5]:
first_row = df.iloc[0]
for col in df.columns:
    print(f"--- {col} ---")
    val = str(first_row[col])
    print(val[:500] if len(val) > 500 else val)
    print()

--- Title ---
Miso-Butter Roast Chicken With Acorn Squash Panzanella

--- Ingredients ---
['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher salt, divided, plus more', '2 small acorn squash (about 3 lb. total)', '2 Tbsp. finely chopped sage', '1 Tbsp. finely chopped rosemary', '6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature', '¼ tsp. ground allspice', 'Pinch of crushed red pepper flakes', 'Freshly ground black pepper', '⅓ loaf good-quality sturdy white bread, torn into 1" pieces (about 2½ cups)', '2 medium apples (such as Gala or Pink Lady; about 14 oz. total), cored, cut

--- Instructions ---
Pat chicken dry with paper towels, season all over with 2 tsp. salt, and tie legs together with kitchen twine. Let sit at room temperature 1 hour.
Meanwhile, halve squash and scoop out seeds. Run a vegetable peeler along ridges of squash halves to remove skin. Cut each half into ½"-thick wedges; arrange on a rimmed baking sheet.
Combine sage, rosemary, and 6 Tbsp. melted butter in a l

## Image Availability\n\nHow many recipes have a matching image file in the Food Images directory?

In [6]:
available_images = set(os.listdir(IMAGE_DIR))
print(f"Total image files on disk: {len(available_images):,}")

df["has_image"] = df["Image_Name"].apply(
    lambda x: isinstance(x, str) and x.strip() in available_images
)
matched = df["has_image"].sum()
print(f"Recipes with matching image: {matched:,} / {len(df):,} ({matched/len(df)*100:.1f}%)")
print(f"Recipes without matching image: {len(df) - matched:,}")

Total image files on disk: 13,582
Recipes with matching image: 0 / 13,501 (0.0%)
Recipes without matching image: 13,501


## Basic Stats: Ingredient Counts per Recipe

In [7]:
def safe_parse(val):
    """Parse a stringified Python list, returning empty list on failure."""
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

df["parsed_ingredients"] = df["Cleaned_Ingredients"].apply(safe_parse)
df["ingredient_count"] = df["parsed_ingredients"].apply(len)

print("Ingredient count statistics:")
print(df["ingredient_count"].describe())
print(f"\nRecipes with 0 ingredients (parse failure or empty): {(df['ingredient_count'] == 0).sum()}")
print(f"\nDistribution of ingredient counts:")
print(df["ingredient_count"].value_counts().sort_index().head(25))

Ingredient count statistics:
count    13501.000000
mean        10.986371
std          4.918713
min          1.000000
25%          7.000000
50%         10.000000
75%         14.000000
max         51.000000
Name: ingredient_count, dtype: float64

Recipes with 0 ingredients (parse failure or empty): 0

Distribution of ingredient counts:
ingredient_count
1       33
2       71
3      241
4      451
5      648
6      946
7     1026
8     1099
9     1246
10    1172
11    1096
12    1031
13     858
14     790
15     647
16     502
17     359
18     307
19     227
20     158
21     145
22     101
23      85
24      70
25      53
Name: count, dtype: int64


In [8]:
# Most common ingredients across all recipes
all_ingredients = [ing for ings in df["parsed_ingredients"] for ing in ings]
print(f"Total ingredient mentions: {len(all_ingredients):,}")
print(f"Unique ingredient strings: {len(set(all_ingredients)):,}")
print(f"\nTop 20 most common ingredients:")
for ing, count in Counter(all_ingredients).most_common(20):
    print(f"  {count:>5,}  {ing}")

Total ingredient mentions: 148,327
Unique ingredient strings: 75,130

Top 20 most common ingredients:
  1,192  Kosher salt
    663  1/2 teaspoon salt
    618  Freshly ground black pepper
    613  1/4 teaspoon salt
    594  Kosher salt, freshly ground pepper
    499  2 tablespoons olive oil
    442  2 large eggs
    402  1 teaspoon vanilla extract
    382  1 teaspoon salt
    374  1 large egg
    338  1/2 cup sugar
    336  1 tablespoon olive oil
    317  1 cup sugar
    305  2 tablespoons fresh lemon juice
    300  1 tablespoon fresh lemon juice
    287  1 teaspoon kosher salt
    285  1/2 teaspoon kosher salt
    283  Nonstick vegetable oil spray
    271  2 tablespoons unsalted butter
    267  1/4 cup olive oil
